# Faster R-CNN Detection Modules

**Purpose:** Shared modules for all Faster R-CNN detection experiments.

**Usage:** Run this notebook first, then use `%run ./FasterRCNN_modules.ipynb` in experiment notebooks.

## Cell 1: Imports & Setup

In [ ]:
# ============================================================================
# Install required dependencies (if not already installed)
# ============================================================================
# This cell will install dependencies if they are missing.
# It checks for multiple critical packages to ensure complete environment.
# ============================================================================

import sys
import warnings
warnings.filterwarnings('ignore')

# Check for all critical dependencies
missing_packages = []

try:
    import torch
except ImportError:
    missing_packages.append('torch')

try:
    import torchvision
except ImportError:
    missing_packages.append('torchvision')

try:
    import cv2
except ImportError:
    missing_packages.append('opencv-python')

try:
    import matplotlib
except ImportError:
    missing_packages.append('matplotlib')

try:
    import seaborn
except ImportError:
    missing_packages.append('seaborn')

try:
    import yaml
except ImportError:
    missing_packages.append('pyyaml')

try:
    import pandas
except ImportError:
    missing_packages.append('pandas')

try:
    import tqdm
except ImportError:
    missing_packages.append('tqdm')

try:
    from PIL import Image
except ImportError:
    missing_packages.append('Pillow')

# Install missing packages if any
if missing_packages:
    print(f"Installing missing dependencies: {', '.join(missing_packages)}")
    print("This may take a few minutes...")
    !{sys.executable} -m pip install {' '.join(missing_packages)} --quiet
    print("✓ Dependencies installed successfully")
else:
    print("✓ All required dependencies already installed")

# Now import all required packages
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models import resnet50
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import json
import csv
import cv2
from PIL import Image

%matplotlib inline

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('Warning: CUDA not available, using CPU')

# Print package versions
print(f'\nPackage Versions:')
print(f'  PyTorch: {torch.__version__}')
print(f'  Torchvision: {torchvision.__version__}')
print(f'  OpenCV: {cv2.__version__}')
print(f'  NumPy: {np.__version__}')

## Cell 2: FasterRCNNDetector Model

In [ ]:
class FasterRCNNDetector(nn.Module):
    """
    Faster R-CNN detector with ResNet50+FPN backbone and customization support.
    """
    
    def __init__(self, num_classes: int = 2, pretrained: bool = True,
                 min_size: int = 640, max_size: int = 640,
                 customize_type: Optional[str] = None,
                 anchor_sizes: Tuple[Tuple[int, ...], ...] = ((4,), (8,), (16,), (32,), (64,)),
                 anchor_aspect_ratios: Tuple[Tuple[float, ...], ...] = ((0.5, 1.0, 2.0),) * 5,
                 rpn_pre_nms_top_n_train: int = 4000,
                 rpn_post_nms_top_n_train: int = 2000,
                 rpn_pre_nms_top_n_test: int = 2000,
                 rpn_post_nms_top_n_test: int = 1000,
                 rpn_nms_thresh: float = 0.85,
                 box_nms_thresh: float = 0.6,
                 rpn_fg_iou_thresh: float = 0.7,
                 rpn_bg_iou_thresh: float = 0.3):
        """
        Initialize Faster R-CNN detector.
        
        Args:
            num_classes: Number of classes (including background)
            pretrained: Use pretrained backbone
            min_size: Minimum image size
            max_size: Maximum image size
            customize_type: Type of customization ('deeper', 'shallower', or None for baseline)
            anchor_sizes: Anchor sizes for each feature map level
            anchor_aspect_ratios: Aspect ratios for anchors at each level
            rpn_pre_nms_top_n_train: Number of proposals before NMS during training
            rpn_post_nms_top_n_train: Number of proposals after NMS during training
            rpn_pre_nms_top_n_test: Number of proposals before NMS during testing
            rpn_post_nms_top_n_test: Number of proposals after NMS during testing
            rpn_nms_thresh: NMS threshold for RPN
            box_nms_thresh: NMS threshold for box predictions
            rpn_fg_iou_thresh: IoU threshold for foreground RPN proposals
            rpn_bg_iou_thresh: IoU threshold for background RPN proposals
        """
        super(FasterRCNNDetector, self).__init__()
        
        self.num_classes = num_classes
        self.customize_type = customize_type
        
        # Load pretrained Faster R-CNN with configurable anchor and RPN parameters
        anchor_generator = AnchorGenerator(
            sizes=anchor_sizes,
            aspect_ratios=anchor_aspect_ratios
        )
        
        self.model = fasterrcnn_resnet50_fpn(
            pretrained=pretrained,
            weights_backbone="DEFAULT" if pretrained else None,

            rpn_anchor_generator=anchor_generator,

            rpn_pre_nms_top_n_train=rpn_pre_nms_top_n_train,
            rpn_post_nms_top_n_train=rpn_post_nms_top_n_train,

            rpn_pre_nms_top_n_test=rpn_pre_nms_top_n_test,
            rpn_post_nms_top_n_test=rpn_post_nms_top_n_test,

            rpn_nms_thresh=rpn_nms_thresh,
            box_nms_thresh=box_nms_thresh,
            
            rpn_fg_iou_thresh=rpn_fg_iou_thresh,
            rpn_bg_iou_thresh=rpn_bg_iou_thresh
        )
        
        # Apply customization before replacing classifier
        if customize_type == 'deeper':
            self._add_conv_layers()
        elif customize_type == 'shallower':
            self._reduce_conv_layers()
        
        # Replace classifier for custom number of classes
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
        
        # Set transform parameters
        self.model.transform.min_size = (min_size,)
        self.model.transform.max_size = max_size
    
    def _add_conv_layers(self):
        """
        V2: Add extra convolutional layers to the ResNet50 backbone after layer2.
        Purpose: Deepen shallow-layer feature extraction for better fine-grained detection.
        
        Strategy: Wrap layer3 with a Sequential that first applies custom conv layers,
        then the original layer3. This preserves FPN's access to layer outputs.
        """
        try:
            # Access the ResNet50 backbone
            backbone = self.model.backbone.body
            
            device = next(self.model.parameters()).device
            
            # Get channel dimensions from layer2 output (should be 512)
            layer2_out_channels = backbone.layer2[-1].conv3.out_channels
            
            # Create new Conv-BN-ReLU blocks
            new_block = nn.Sequential(
                nn.Conv2d(layer2_out_channels, layer2_out_channels, kernel_size=3, 
                         stride=1, padding=1, bias=False),
                nn.BatchNorm2d(layer2_out_channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(layer2_out_channels, layer2_out_channels, kernel_size=3,
                         stride=1, padding=1, bias=False),
                nn.BatchNorm2d(layer2_out_channels),
                nn.ReLU(inplace=True)
            ).to(device)
            
            # Store original layer3 and layer4
            original_layer3 = backbone.layer3
            original_layer4 = backbone.layer4
            
            # Replace layer3 with: custom_conv -> original_layer3
            # This way FPN can still access "layer3" output
            backbone.layer3 = nn.Sequential(
                new_block,
                original_layer3
            )
            
            print("✅ Added 2 convolutional layers to backbone after layer2 (deeper model)")
            print(f"   Layer structure preserved for FPN compatibility")
            
        except Exception as e:
            print(f"⚠️ Warning: Could not add conv layers: {e}")
            import traceback
            traceback.print_exc()
            print("Continuing with standard model...")
    
    def _reduce_conv_layers(self):
        """
        V3: Reduce convolutional layers in the ResNet50 backbone.
        Purpose: Create a lighter model with faster inference and reduced overfitting.
        Strategy: Remove some bottleneck blocks from layer3.
        """
        try:
            backbone = self.model.backbone.body
            
            # Standard ResNet50 layer3 has 6 bottleneck blocks
            # We'll reduce it to 3 blocks (remove 3 bottlenecks = 9 conv layers)
            original_num_blocks = len(backbone.layer3)
            new_num_blocks = max(1, original_num_blocks // 2)
            
            # Create new layer3 with fewer blocks
            # Get configuration from first block
            first_block = backbone.layer3[0]
            
            # Extract key parameters
            inplanes = first_block.conv1.in_channels
            planes = first_block.conv1.out_channels
            stride = first_block.downsample[0].stride if first_block.downsample else (1, 1)
            
            # Rebuild layer3 with reduced blocks
            new_layer3 = nn.Sequential()
            
            # First block might have downsampling
            for i in range(new_num_blocks):
                block_stride = stride if i == 0 else 1
                block = type(first_block)(
                    inplanes if i == 0 else planes * first_block.expansion,
                    planes,
                    stride=block_stride,
                    downsample=first_block.downsample if i == 0 else None
                )
                new_layer3.add_module(str(i), block)
            
            # Replace layer3
            backbone.layer3 = new_layer3
            
            print(f"✅ Reduced layer3 from {original_num_blocks} to {new_num_blocks} bottleneck blocks")
            print(f"✅ Removed {original_num_blocks - new_num_blocks} blocks ({(original_num_blocks - new_num_blocks) * 3} conv layers)")
            print("✅ Shallower backbone configured successfully")
            
        except Exception as e:
            print(f"⚠️ Warning: Could not reduce conv layers: {e}")
            print("Continuing with standard model...")
    
    def forward(self, images: List[torch.Tensor], 
                targets: List[Dict[str, torch.Tensor]] = None):
        """
        Forward pass.
        
        Args:
            images: List of input tensors
            targets: Target dictionaries (training only)
            
        Returns:
            Training: dict with losses
            Inference: list of detections
        """
        return self.model(images, targets)
    
    def save(self, save_path: str):
        """Save model."""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'num_classes': self.num_classes,
            'customize_type': self.customize_type
        }, save_path)
    
    def load(self, load_path: str):
        """Load model weights."""
        checkpoint = torch.load(load_path, map_location='cpu')
        self.model.load_state_dict(checkpoint['model_state_dict'])


print("✓ FasterRCNNDetector class defined successfully")

## Cell 3: FasterRCNNTrainer

In [ ]:
class FasterRCNNTrainer:
    """
    Enhanced trainer for Faster R-CNN models with CSV logging and checkpointing.
    """
    
    def __init__(self, config: Dict[str, any]):
        """
        Initialize Faster R-CNN trainer with complete training configuration.
        
        Args:
            config: Training configuration dictionary containing:
                - learning_rate: Initial learning rate
                - weight_decay: L2 regularization
                - epochs: Number of training epochs
                - patience: Early stopping patience
                - optimizer: Optimizer type (currently only 'adam' supported)
        """
        self.learning_rate = config['learning_rate']
        self.weight_decay = config['weight_decay']
        self.epochs = config['epochs']
        self.patience = config['patience']
        self.optimizer_type = config.get('optimizer', 'adam')
        self.config = config  # Store config for checkpointing
    
    def train(self, model, train_loader, val_loader, output_dir: str) -> Dict:
        """
        Train Faster R-CNN model with CSV logging and checkpointing.
        
        Args:
            model: FasterRCNNDetector instance
            train_loader: Training data loader
            val_loader: Validation data loader
            output_dir: Directory to save outputs
            
        Returns:
            Training history dictionary
        """
        print("=" * 80)
        print("FASTER R-CNN TRAINING")
        print("=" * 80)
        
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.model.to(device)
        
        # Setup optimizer
        params = [p for p in model.parameters() if p.requires_grad]
        if self.optimizer_type == 'adam':
            optimizer = torch.optim.Adam(params, lr=self.learning_rate, 
                                        weight_decay=self.weight_decay)
        elif self.optimizer_type == 'adamw':
            optimizer = torch.optim.AdamW(params, lr=self.learning_rate, 
                                         weight_decay=self.weight_decay)
        elif self.optimizer_type == 'sgd':
            optimizer = torch.optim.SGD(
                params,
                lr=self.learning_rate,
                momentum=0.9,
                weight_decay=self.weight_decay
                )
        else:
            raise ValueError(f"Invalid optimizer type: {self.optimizer_type}")
       
        # Setup learning rate scheduler (Cosine Annealing)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=self.epochs,
            eta_min=1e-6
        )
        
        # Check for existing checkpoint and resume if available
        checkpoint = self._load_checkpoint(output_path)
        start_epoch = 0
        best_map50 = 0.0
        early_stop_counter = 0
        best_train_loss = float('inf')
        
        if checkpoint:
            start_epoch = checkpoint['epoch'] + 1
            model.model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            best_map50 = checkpoint['best_map50']
            early_stop_counter = checkpoint.get('early_stop_counter', 0)
            best_train_loss = checkpoint.get('best_train_loss', float('inf'))
            print(f'Resumed from epoch {start_epoch}, best mAP@0.5: {best_map50:.4f}')
            print(f'Early stop counter: {early_stop_counter}/{self.patience}')
            print(f'Best training loss: {best_train_loss:.4f}')
        
        # Setup CSV logging with core metrics
        csv_path = output_path / 'training_history.csv'
        
        # If resuming, append to existing CSV, else create new
        if checkpoint and csv_path.exists():
            csv_file = open(csv_path, 'a', newline='')
            csv_writer = csv.writer(csv_file)
            print(f'Appending to existing CSV: {csv_path}')
        else:
            csv_file = open(csv_path, 'w', newline='')
            csv_writer = csv.writer(csv_file)
            csv_writer.writerow([
                'epoch', 
                'train_loss', 
                'train_box_loss', 'train_cls_loss',
                'precision', 'recall', 'f1_score',
                'map50', 'map50_95',
                'learning_rate'
            ])
        
        # Setup mixed precision training (AMP) for faster training
        use_amp = torch.cuda.is_available()
        scaler = GradScaler(enabled=use_amp)
        
        # Print model architecture summary
        print(f"\nModel Architecture:")
        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Total trainable parameters: {total_params:,}")
        
        print(f"Training configuration:")
        print(f"  Epochs: {self.epochs} (starting from {start_epoch})")
        print(f"  Learning rate: {self.learning_rate}")
        print(f"  Weight decay: {self.weight_decay}")
        print(f"  Device: {device}")
        print(f"  Early stopping patience: {self.patience}")
        print(f"  Mixed precision (AMP): {use_amp}")
        print(f"  CSV logging: {csv_path}")
        print(f"  Checkpoint saving: Every epoch")
        
        for epoch in range(start_epoch, self.epochs):
            # Training epoch
            model.model.train()
            epoch_loss = 0.0
            epoch_box_loss = 0.0
            epoch_cls_loss = 0.0
            batch_count = 0
            
            train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.epochs} [Train]", dynamic_ncols=True, leave=True)
            
            for batch_idx, (images, targets) in enumerate(train_pbar):
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                # Forward pass with AMP (mixed precision)
                with autocast(enabled=use_amp):
                    loss_dict = model(images, targets)
                    losses = sum(loss for loss in loss_dict.values())
                
                # Skip if total loss is NaN or Inf (safety check)
                if not torch.isfinite(losses):
                    print(f"  ⚠ Warning: Non-finite loss at batch {batch_idx}, skipping")
                    continue
                
                # Track box and cls loss components
                epoch_box_loss += loss_dict.get('loss_box_reg', 0).item()
                epoch_cls_loss += loss_dict.get('loss_classifier', 0).item()
                
                # Backward pass with gradient scaling (AMP)
                optimizer.zero_grad()
                scaler.scale(losses).backward()
                
                # Gradient clipping to prevent explosion
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                
                # Optimizer step with scaler
                scaler.step(optimizer)
                scaler.update()
                
                epoch_loss += losses.item()
                batch_count += 1
                
                # Update progress bar with current metrics
                avg_loss = epoch_loss / batch_count
                avg_box = epoch_box_loss / batch_count
                avg_cls = epoch_cls_loss / batch_count
                train_pbar.set_postfix({
                    'loss': f'{avg_loss:.3f}',
                    'box': f'{avg_box:.3f}',
                    'cls': f'{avg_cls:.3f}'
                })
            
            train_pbar.close()
            
            # Calculate average losses
            avg_train_loss = epoch_loss / max(batch_count, 1)
            avg_box_loss = epoch_box_loss / max(batch_count, 1)
            avg_cls_loss = epoch_cls_loss / max(batch_count, 1)

            if avg_train_loss < best_train_loss:
                best_train_loss = avg_train_loss
            
            # Fast mAP evaluation on validation set (< 5 seconds)
            map50, map50_95, precision, recall = self._fast_evaluate(model, val_loader, device, epoch, self.epochs)
            
            # Log to CSV with core metrics
            current_lr = scheduler.get_last_lr()[0]
            f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)
            csv_writer.writerow([
                epoch + 1, 
                f'{avg_train_loss:.4f}', 
                f'{avg_box_loss:.4f}',
                f'{avg_cls_loss:.4f}',
                f'{precision:.4f}',
                f'{recall:.4f}',
                f'{f1_score:.4f}',
                f'{map50:.4f}',
                f'{map50_95:.4f}',
                f'{current_lr:.8f}'
            ])
            csv_file.flush()
            
            # Step the learning rate scheduler after each epoch
            scheduler.step()
            
            # Checkpoint and early stopping
            if map50 > best_map50:
                best_map50 = map50
                early_stop_counter = 0

                torch.save(model.model.state_dict(), output_path / 'best_model.pth')
                print(f'  New best model saved (mAP@0.5: {map50:.3f})')
            else:
                early_stop_counter += 1

            # Save checkpoint every 1 epochs
            self._save_checkpoint(epoch, model, optimizer, scheduler, best_map50, early_stop_counter, best_train_loss, output_path)
            
            # Early stop
            if early_stop_counter >= self.patience:
                print(f'\nEarly stopping triggered at epoch {epoch+1}')
                break
        
        csv_file.close()
        
        # Generate training curve plots
        self._plot_training_curves(csv_path, output_path)
        
        history = {
            'best_loss': best_train_loss,
            'total_epochs': epoch + 1,
            'early_stopped': early_stop_counter >= self.patience
        }
        
        print(f"\nTraining completed!")
        print(f"  Best training loss: {best_train_loss:.4f}")
        print(f"  Total epochs: {history['total_epochs']}")
        print(f"  Early stopped: {history['early_stopped']}")
        
        return history
    
    def _save_checkpoint(self, epoch, model, optimizer, scheduler, best_map50, early_stop_counter, best_train_loss, output_path):
        """
        Save training checkpoint every 1 epochs and always save latest.
        """
        checkpoint_dir = output_path / 'checkpoints'
        checkpoint_dir.mkdir(exist_ok=True)
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_map50': best_map50,
            'early_stop_counter': early_stop_counter,
            'best_train_loss': best_train_loss,
            'config': self.config
        }
        
        # Save every 1 epochs. For more frequent saving, adjust the condition below.
        # if (epoch + 1) % 10 == 0:
        # checkpoint_path = checkpoint_dir / f'checkpoint_epoch_{epoch+1}.pth'
        # torch.save(checkpoint, checkpoint_path)
        # print(f'  Periodic checkpoint saved: checkpoint_epoch_{epoch+1}.pth')
        
        # Always save latest
        latest_path = checkpoint_dir / 'latest_checkpoint.pth'
        torch.save(checkpoint, latest_path)
    
    def _load_checkpoint(self, output_path):
        """
        Load latest checkpoint if available, with configuration validation.
        """
        checkpoint_dir = output_path / 'checkpoints'
        latest_path = checkpoint_dir / 'latest_checkpoint.pth'
        
        if not latest_path.exists():
            return None
        
        print(f'Found existing checkpoint: {latest_path}')
        checkpoint = torch.load(latest_path, map_location='cpu')
        
        # Configuration validation
        saved_config = checkpoint.get('config', {})
        current_config = self.config
        
        # Check for differences
        differences = []
        for key in set(saved_config.keys()) | set(current_config.keys()):
            if key not in saved_config:
                differences.append(f'{key}: missing in saved config (current: {current_config[key]})')
            elif key not in current_config:
                differences.append(f'{key}: missing in current config (saved: {saved_config[key]})')
            elif saved_config[key] != current_config[key]:
                differences.append(f'{key}: saved={saved_config[key]}, current={current_config[key]}')
        
        if differences:
            print('⚠️ Warning: Configuration differences detected:')
            for diff in differences:
                print(f'  {diff}')
            print('Continuing with current configuration...')
        
        return checkpoint
    
    def _fast_evaluate(self, model, val_loader, device, epoch=0, epochs=1):
        """
        Fast mAP evaluation (< 5 seconds) using simplified IoU matching.
        """
        all_preds = []
        all_gts = []
        
        eval_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Eval]", dynamic_ncols=True, leave=True)
        
        model.model.eval()
        with torch.no_grad():
            for images, targets in eval_pbar:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                if all(len(t['boxes']) == 0 for t in targets):
                    continue
                
                predictions = model(images)
                
                all_preds.extend(predictions)
                all_gts.extend(targets)
                
                eval_pbar.set_postfix({'batch': len(images)})

        model.model.train()

        if len(all_preds) == 0 or len(all_gts) == 0:
            return 0.0, 0.0, 0.0, 0.0
        
        map50, map50_95, precision, recall = self._compute_fast_map(all_preds, all_gts)
        
        eval_pbar.close()
        
        print(f"Epoch {epoch+1}/{epochs} [Eval] completed: P={precision:.3f}, R={recall:.3f}, mAP@0.5={map50:.3f}, mAP@0.5:0.95={map50_95:.3f}")
        
        return map50, map50_95, precision, recall
    
    def _compute_fast_map(self, predictions, ground_truths, iou_thresholds=[0.5, 0.75], score_threshold=0.3):
        """
        Compute approximate mAP using simplified IoU matching with score filtering.
        
        Args:
            predictions: List of prediction dictionaries containing 'boxes', 'scores', 'labels'
            ground_truths: List of ground truth dictionaries containing 'boxes', 'labels'
            iou_thresholds: IoU thresholds for evaluation (default: [0.5, 0.75])
            score_threshold: Minimum confidence score to keep a prediction (default: 0.3)
        """
        total_tp_50 = 0
        total_fp_50 = 0
        total_fn_50 = 0
        
        total_tp_75 = 0
        total_fp_75 = 0
        total_fn_75 = 0
        
        for pred, gt in zip(predictions, ground_truths):
            # Apply score filtering to remove low-confidence predictions
            scores = pred['scores'].cpu().numpy()
            keep_mask = scores >= score_threshold
            pred_boxes = pred['boxes'][keep_mask].cpu().numpy()
            
            gt_boxes = gt['boxes'].cpu().numpy()
            
            if len(pred_boxes) == 0 or len(gt_boxes) == 0:
                total_fn_50 += len(gt_boxes)
                total_fn_75 += len(gt_boxes)
                continue
            
            ious = self._compute_iou_matrix(pred_boxes, gt_boxes)
            
            tp_50, fp_50, fn_50 = self._match_predictions(ious, scores[keep_mask], threshold=0.5)
            total_tp_50 += tp_50
            total_fp_50 += fp_50
            total_fn_50 += fn_50
            
            tp_75, fp_75, fn_75 = self._match_predictions(ious, scores[keep_mask], threshold=0.75)
            total_tp_75 += tp_75
            total_fp_75 += fp_75
            total_fn_75 += fn_75
        
        precision_50 = total_tp_50 / max(total_tp_50 + total_fp_50, 1)
        recall_50 = total_tp_50 / max(total_tp_50 + total_fn_50, 1)
        
        precision_75 = total_tp_75 / max(total_tp_75 + total_fp_75, 1)
        recall_75 = total_tp_75 / max(total_tp_75 + total_fn_75, 1)
        
        ap_50 = precision_50 * recall_50
        ap_75 = precision_75 * recall_75
        
        map50 = ap_50
        map50_95 = (ap_50 + ap_75) / 2.0
        
        precision = precision_50
        recall = recall_50
        
        return float(map50), float(map50_95), float(precision), float(recall)
    
    def _match_predictions(self, ious, scores, threshold=0.5):
        """
        Match predictions to ground truth boxes based on IoU threshold and confidence scores.
        
        Args:
            ious: IoU matrix between predictions and ground truths
            scores: Confidence scores for predictions
            threshold: IoU threshold for matching
        """
        n_pred, n_gt = ious.shape
        
        if n_pred == 0 or n_gt == 0:
            return 0, n_pred, n_gt
        
        matched_gt = np.zeros(n_gt, dtype=bool)
        matched_pred = np.zeros(n_pred, dtype=bool)
        
        # Sort predictions by confidence score (descending) for priority matching
        sorted_pred_indices = np.argsort(scores)[::-1]
        
        matches = []
        for pred_idx in sorted_pred_indices:
            for j in range(n_gt):
                if ious[pred_idx, j] >= threshold:
                    matches.append((ious[pred_idx, j], pred_idx, j))
        
        # Sort by IoU (descending) for greedy matching
        matches.sort(reverse=True)
        
        tp = 0
        for iou_val, pred_idx, gt_idx in matches:
            if not matched_pred[pred_idx] and not matched_gt[gt_idx]:
                matched_pred[pred_idx] = True
                matched_gt[gt_idx] = True
                tp += 1
        
        fp = n_pred - tp
        fn = n_gt - tp
        
        return tp, fp, fn
    
    def _compute_iou_matrix(self, boxes1, boxes2):
        """
        Compute pairwise IoU between two sets of boxes.
        """
        N = len(boxes1)
        M = len(boxes2)
        
        if N == 0 or M == 0:
            return np.zeros((N, M))
        
        inter_x1 = np.maximum(boxes1[:, 0][:, np.newaxis], boxes2[:, 0][np.newaxis, :])
        inter_y1 = np.maximum(boxes1[:, 1][:, np.newaxis], boxes2[:, 1][np.newaxis, :])
        inter_x2 = np.minimum(boxes1[:, 2][:, np.newaxis], boxes2[:, 2][np.newaxis, :])
        inter_y2 = np.minimum(boxes1[:, 3][:, np.newaxis], boxes2[:, 3][np.newaxis, :])
        
        inter_w = np.maximum(0, inter_x2 - inter_x1)
        inter_h = np.maximum(0, inter_y2 - inter_y1)
        inter_area = inter_w * inter_h
        
        area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
        area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
        union_area = area1[:, np.newaxis] + area2[np.newaxis, :] - inter_area
        
        iou = inter_area / (union_area + 1e-6)
        
        return iou
    
    def _plot_training_curves(self, csv_path: Path, output_path: Path):
        """
        Generate simplified training curve plots (only 2 core charts).
        """
        try:
            df = pd.read_csv(csv_path)
            
            if len(df) == 0:
                print("⚠ Warning: No training data to plot")
                return
            
            # Plot 1: Training Loss Curves
            fig, ax1 = plt.subplots(figsize=(10, 6))
            
            ax1.plot(df['epoch'], df['train_loss'], 'b-o', label='Total Train Loss', linewidth=2, markersize=4)
            if 'train_box_loss' in df.columns:
                ax1.plot(df['epoch'], df['train_box_loss'], 'g-s', label='Box Loss', linewidth=2, markersize=4)
            if 'train_cls_loss' in df.columns:
                ax1.plot(df['epoch'], df['train_cls_loss'], 'm-^', label='Cls Loss', linewidth=2, markersize=4)
            
            ax1.set_xlabel('Epoch', fontsize=12)
            ax1.set_ylabel('Loss', fontsize=12)
            ax1.set_title('Training Loss Components', fontsize=13, fontweight='bold')
            ax1.legend(loc='best', fontsize=11)
            ax1.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(output_path / 'loss_curve.png', dpi=150, bbox_inches='tight')
            plt.close()
            print(f"✓ Loss curve saved to: {output_path / 'loss_curve.png'}")
            
            # Plot 2: mAP Curve (most important!)
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(df['epoch'], df['map50'], 'g-o', label='mAP@0.5', linewidth=2, markersize=6)
            ax.plot(df['epoch'], df['map50_95'], 'b-s', label='mAP@0.5:0.95', linewidth=2, markersize=6)
            
            if 'precision' in df.columns and df['precision'].max() > 0:
                ax.plot(df['epoch'], df['precision'], 'm-^', label='Precision', 
                       linewidth=2, markersize=4, alpha=0.7)
            if 'recall' in df.columns and df['recall'].max() > 0:
                ax.plot(df['epoch'], df['recall'], 'c-d', label='Recall', 
                       linewidth=2, markersize=4, alpha=0.7)
            
            ax.set_xlabel('Epoch', fontsize=12)
            ax.set_ylabel('Metric Value', fontsize=12)
            ax.set_title('Core Performance Metrics', fontsize=14, fontweight='bold')
            ax.legend(loc='best', fontsize=11)
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(output_path / 'map_curve.png', dpi=150, bbox_inches='tight')
            plt.close()
            print(f"✓ mAP curve saved to: {output_path / 'map_curve.png'}")
        
        except Exception as e:
            print(f"⚠ Warning: Failed to generate training plots: {e}")


print("✓ FasterRCNNTrainer class defined successfully")

## Cell 4: DetectionEvaluator

In [ ]:
class DetectionEvaluator:
    """
    Evaluator for Faster R-CNN detection models.
    """
    
    def __init__(self):
        self.metrics = {}
    
    def evaluate_fasterrcnn(self, model, test_loader, output_dir, class_names):
        """
        Evaluate Faster R-CNN model on test dataset.
        
        Args:
            model: FasterRCNNDetector instance
            test_loader: Test data loader
            output_dir: Directory to save results
            class_names: List of class names
            
        Returns:
            dict: Evaluation metrics including mAP, precision, recall, f1_score
        """
        print("Running evaluation on test set...")
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.model.to(device)
        model.model.eval()
        
        all_preds = []
        all_gts = []
        
        with torch.no_grad():
            for images, targets in tqdm(test_loader, desc="Evaluating"):
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                predictions = model(images)
                
                all_preds.extend(predictions)
                all_gts.extend(targets)
        
        # Calculate metrics using simplified mAP
        trainer = FasterRCNNTrainer({'learning_rate': 0.001, 'weight_decay': 1e-4, 'epochs': 1, 'patience': 1})
        map50, map50_95, precision, recall = trainer._compute_fast_map(all_preds, all_gts)
        
        # Calculate F1-Score
        f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)
        
        metrics = {
            'mAP50': float(map50),
            'mAP50_95': float(map50_95),
            'precision': float(precision),
            'recall': float(recall),
            'f1_score': float(f1_score),
            'num_test_samples': len(all_gts)
        }
        
        # Save metrics
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        metrics_path = output_path / 'evaluation_metrics.json'
        with open(metrics_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        
        # Generate and save confusion matrix
        print("\nGenerating confusion matrix...")
        num_classes = len(class_names)
        cm = self._compute_confusion_matrix(all_preds, all_gts, num_classes)
        self._save_confusion_matrix(cm, class_names, str(output_path))
        
        # Generate and save PR curves data
        print("\nGenerating PR curves data (per-class)...")
        pr_data = self._compute_per_class_pr_curves(all_preds, all_gts, num_classes, class_names)
        pr_path = output_path / 'pr_curves_data.json'
        with open(pr_path, 'w') as f:
            json.dump(pr_data, f, indent=2)
        print(f"✓ PR curves data saved to: {pr_path}")
        
        print(f"\nEvaluation completed:")
        print(f"  mAP@0.5: {metrics['mAP50']:.4f}")
        print(f"  mAP@0.5:0.95: {metrics['mAP50_95']:.4f}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall: {metrics['recall']:.4f}")
        print(f"  F1-Score: {metrics['f1_score']:.4f}")
        print(f"  Saved to: {metrics_path}")
        
        return metrics
    
    def plot_detection_results(self, model, test_loader, num_samples=5):
        """
        Plot detection results with bounding boxes inline.
        Returns matplotlib figure for inline display.
        """
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.model.to(device)
        model.model.eval()
        
        fig, axes = plt.subplots(1, num_samples, figsize=(4*num_samples, 4))
        if num_samples == 1:
            axes = [axes]
        
        count = 0
        for images, targets in test_loader:
            if count >= num_samples:
                break
            
            img_tensor = images[0].to(device)
            
            with torch.no_grad():
                predictions = model([img_tensor])
            
            # Convert tensor to numpy for plotting
            img_np = img_tensor.cpu().permute(1, 2, 0).numpy()
            img_np = (img_np * 255).astype(np.uint8)
            
            ax = axes[count]
            ax.imshow(img_np)
            
            # Draw predictions
            pred_boxes = predictions[0]['boxes'].cpu().numpy()
            pred_scores = predictions[0]['scores'].cpu().numpy()
            pred_labels = predictions[0]['labels'].cpu().numpy()
            
            for box, score, label in zip(pred_boxes, pred_scores, pred_labels):
                if score > 0.5:  # Only show high confidence detections
                    x1, y1, x2, y2 = box.astype(int)
                    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                                        linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, y1-5, f'{score:.2f}', color='red', fontsize=8)
            
            ax.set_title(f'Sample {count+1}', fontsize=10)
            ax.axis('off')
            count += 1
        
        plt.tight_layout()
        plt.suptitle('Detection Results (Red Boxes = Predictions)', fontsize=14, fontweight='bold', y=1.02)
        
        return fig
    
    def plot_training_losses(self, output_dir):
        """
        Plot training loss curves by loading saved loss_curve.png image.
        Returns matplotlib figure for inline display.
        """
        loss_curve_path = Path(output_dir) / 'training' / 'loss_curve.png'
        
        if not loss_curve_path.exists():
            print(f"Warning: loss_curve.png not found: {loss_curve_path}")
            return None
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Load and display the saved image
        loss_img = plt.imread(loss_curve_path)
        ax.imshow(loss_img)
        ax.axis('off')
        ax.set_title('Training Loss Curves', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        return fig
    
    def plot_validation_mAP(self, output_dir):
        """
        Plot validation mAP curves by loading saved map_curve.png image.
        Returns matplotlib figure for inline display.
        """
        map_curve_path = Path(output_dir) / 'training' / 'map_curve.png'
        
        if not map_curve_path.exists():
            print(f"Warning: map_curve.png not found: {map_curve_path}")
            return None
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Load and display the saved image
        map_img = plt.imread(map_curve_path)
        ax.imshow(map_img)
        ax.axis('off')
        ax.set_title('Validation mAP Curves', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        return fig
    
        axes[1].axis('off')
        axes[1].set_title('Validation mAP Curves', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        return fig
    
    def plot_confusion_matrix(self, output_dir):
        """
        从 evaluation 目录加载并显示混淆矩阵
        
        Args:
            output_dir: evaluation 目录路径
        
        Returns:
            matplotlib figure 或 None
        """
        output_path = Path(output_dir)
        cm_path = output_path / 'confusion_matrix.png'
        cm_norm_path = output_path / 'confusion_matrix_normalized.png'
        
        if not cm_path.exists() or not cm_norm_path.exists():
            print(f"Warning: Confusion matrix images not found in {output_dir}")
            return None
        
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # 未归一化
        cm_img = plt.imread(cm_path)
        axes[0].imshow(cm_img)
        axes[0].axis('off')
        axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
        
        # 归一化
        cm_norm_img = plt.imread(cm_norm_path)
        axes[1].imshow(cm_norm_img)
        axes[1].axis('off')
        axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        return fig
    
    def plot_pr_curve(self, output_dir):
        """
        从 training_history.csv 绘制 Precision-Recall 曲线
        
        Args:
            output_dir: 输出目录路径
        
        Returns:
            matplotlib figure 或 None
        """
        csv_path = Path(output_dir) / 'training' / 'training_history.csv'
        
        if not csv_path.exists():
            print(f"Warning: CSV file not found: {csv_path}")
            return None
        
        df = pd.read_csv(csv_path)
        
        if 'precision' not in df.columns or 'recall' not in df.columns:
            print("Warning: Precision or Recall columns not found in CSV")
            return None
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.plot(df['recall'], df['precision'], 'b-o', linewidth=2, markersize=4)
        ax.set_xlabel('Recall', fontsize=12)
        ax.set_ylabel('Precision', fontsize=12)
        ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # 添加 PR-AUC 计算（可选）
        try:
            from sklearn.metrics import auc
            pr_auc = auc(df['recall'], df['precision'])
            ax.text(0.05, 0.95, f'PR-AUC: {pr_auc:.4f}', 
                   transform=ax.transAxes, fontsize=12, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        except ImportError:
            print("Note: scikit-learn not available, skipping PR-AUC calculation")
        
        plt.tight_layout()
        return fig
    
    def plot_f1_curve(self, output_dir):
        """
        从 training_history.csv 绘制 F1-Score 曲线
        
        Args:
            output_dir: 输出目录路径
        
        Returns:
            matplotlib figure 或 None
        """
        csv_path = Path(output_dir) / 'training' / 'training_history.csv'
        
        if not csv_path.exists():
            print(f"Warning: CSV file not found: {csv_path}")
            return None
        
        df = pd.read_csv(csv_path)
        
        if 'f1_score' not in df.columns:
            print("Warning: f1_score column not found in CSV")
            return None
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.plot(df['epoch'], df['f1_score'], 'm-o', linewidth=2, markersize=6)
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('F1-Score', fontsize=12)
        ax.set_title('F1-Score Curve', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # 标注最佳 F1
        best_idx = df['f1_score'].idxmax()
        best_epoch = int(df.loc[best_idx, 'epoch'])
        best_f1 = df['f1_score'].max()
        ax.annotate(f'Best: {best_f1:.4f}\n@ Epoch {best_epoch}',
                   xy=(best_epoch, best_f1),
                   xytext=(10, 10), textcoords='offset points',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.5),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
        
        plt.tight_layout()
        return fig
    
    def _compute_iou_matrix(self, boxes1, boxes2):
        """
        Compute pairwise IoU between two sets of boxes.
        """
        N = len(boxes1)
        M = len(boxes2)
        
        if N == 0 or M == 0:
            return np.zeros((N, M))
        
        inter_x1 = np.maximum(boxes1[:, 0][:, np.newaxis], boxes2[:, 0][np.newaxis, :])
        inter_y1 = np.maximum(boxes1[:, 1][:, np.newaxis], boxes2[:, 1][np.newaxis, :])
        inter_x2 = np.minimum(boxes1[:, 2][:, np.newaxis], boxes2[:, 2][np.newaxis, :])
        inter_y2 = np.minimum(boxes1[:, 3][:, np.newaxis], boxes2[:, 3][np.newaxis, :])
        
        inter_w = np.maximum(0, inter_x2 - inter_x1)
        inter_h = np.maximum(0, inter_y2 - inter_y1)
        inter_area = inter_w * inter_h
        
        area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
        area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
        union_area = area1[:, np.newaxis] + area2[np.newaxis, :] - inter_area
        
        iou = inter_area / (union_area + 1e-6)
        
        return iou
    
    def _compute_confusion_matrix(self, predictions, ground_truths, num_classes):
        """
        生成混淆矩阵数据
        
        Args:
            predictions: 预测结果列表
            ground_truths: 真实标签列表
            num_classes: 类别数量（不含背景）
        
        Returns:
            confusion_matrix: numpy array (num_classes+1, num_classes+1)
            - 行：预测类别（0=背景，1~num_classes=实际类别）
            - 列：真实类别
        """
        cm = np.zeros((num_classes + 1, num_classes + 1), dtype=np.int32)
        
        for pred, gt in zip(predictions, ground_truths):
            pred_boxes = pred['boxes'].cpu().numpy()
            pred_labels = pred['labels'].cpu().numpy()
            pred_scores = pred['scores'].cpu().numpy()
            
            gt_boxes = gt['boxes'].cpu().numpy()
            gt_labels = gt['labels'].cpu().numpy()
            
            if len(pred_boxes) == 0 and len(gt_boxes) == 0:
                continue
            
            # 使用 IoU matching
            if len(pred_boxes) > 0 and len(gt_boxes) > 0:
                ious = self._compute_iou_matrix(pred_boxes, gt_boxes)
                
                matched_gt = np.zeros(len(gt_boxes), dtype=bool)
                matched_pred = np.zeros(len(pred_boxes), dtype=bool)
                
                # 按 IoU 降序匹配
                matches = []
                for i in range(len(pred_boxes)):
                    for j in range(len(gt_boxes)):
                        if ious[i, j] >= 0.5:
                            matches.append((ious[i, j], i, j))
                matches.sort(reverse=True)
                
                for iou_val, pred_idx, gt_idx in matches:
                    if not matched_pred[pred_idx] and not matched_gt[gt_idx]:
                        matched_pred[pred_idx] = True
                        matched_gt[gt_idx] = True
                        
                        pred_class = int(pred_labels[pred_idx])
                        gt_class = int(gt_labels[gt_idx])
                        cm[pred_class, gt_class] += 1
                
                # 未匹配的预测（False Positives）
                for i in range(len(pred_boxes)):
                    if not matched_pred[i]:
                        pred_class = int(pred_labels[i])
                        cm[pred_class, 0] += 1
                
                # 未匹配的真实标签（False Negatives）
                for j in range(len(gt_boxes)):
                    if not matched_gt[j]:
                        gt_class = int(gt_labels[j])
                        cm[0, gt_class] += 1
            elif len(gt_boxes) > 0:
                # 只有真实标签，没有预测
                for j in range(len(gt_boxes)):
                    gt_class = int(gt_labels[j])
                    cm[0, gt_class] += 1
            elif len(pred_boxes) > 0:
                # 只有预测，没有真实标签
                for i in range(len(pred_boxes)):
                    pred_class = int(pred_labels[i])
                    cm[pred_class, 0] += 1
        
        return cm
    
    def _save_confusion_matrix(self, cm, class_names, output_dir):
        """
        保存混淆矩阵图片（未归一化 + 归一化）
        
        Args:
            cm: 混淆矩阵 (numpy array)
            class_names: 类别名称列表（不含背景）
            output_dir: 输出目录
        """
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        # 类别标签（包含背景）
        labels = ['Background'] + class_names
        
        # 1. 未归一化混淆矩阵
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        ax.figure.colorbar(im, ax=ax)
        
        ax.set(xticks=np.arange(len(labels)),
               yticks=np.arange(len(labels)),
               xticklabels=labels, yticklabels=labels,
               ylabel='True label',
               xlabel='Predicted label',
               title='Confusion Matrix (Counts)')
        
        # 添加数值标签
        thresh = cm.max() / 2.
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, format(cm[i, j], 'd'),
                       ha="center", va="center",
                       color="white" if cm[i, j] > thresh else "black")
        
        plt.tight_layout()
        plt.savefig(output_path / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Confusion matrix saved to: {output_path / 'confusion_matrix.png'}")
        
        # 2. 归一化混淆矩阵
        cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-6)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm_normalized, interpolation='nearest', cmap=plt.cm.Blues)
        ax.figure.colorbar(im, ax=ax)
        
        ax.set(xticks=np.arange(len(labels)),
               yticks=np.arange(len(labels)),
               xticklabels=labels, yticklabels=labels,
               ylabel='True label',
               xlabel='Predicted label',
               title='Confusion Matrix (Normalized)')
        
        # 添加百分比标签
        thresh = cm_normalized.max() / 2.
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, format(cm_normalized[i, j], '.2f'),
                       ha="center", va="center",
                       color="white" if cm_normalized[i, j] > thresh else "black")
        
        plt.tight_layout()
        plt.savefig(output_path / 'confusion_matrix_normalized.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Normalized confusion matrix saved to: {output_path / 'confusion_matrix_normalized.png'}")
    
    def _compute_per_class_pr_curves(self, predictions, ground_truths, num_classes, class_names):
        """
        计算每个类别的 PR 曲线数据
        
        Args:
            predictions: 预测结果列表
            ground_truths: 真实标签列表
            num_classes: 类别数量
            class_names: 类别名称列表
        
        Returns:
            pr_data: dict - 包含每个类别和综合的 PR 数据
        """
        import numpy as np
        from sklearn.metrics import auc
        
        # 定义置信度阈值数组
        conf_thresholds = np.arange(0.0, 1.05, 0.05)
        
        # 初始化 PR 数据结构
        pr_data = {
            'all_classes': {'precision': [], 'recall': [], 'ap': 0.0}
        }
        for i in range(num_classes):
            pr_data[f'class_{i}'] = {'precision': [], 'recall': [], 'ap': 0.0}
        
        # 对每个阈值计算 Precision 和 Recall
        for threshold in conf_thresholds:
            all_tp = 0
            all_fp = 0
            all_fn = 0
            
            class_tp = np.zeros(num_classes)
            class_fp = np.zeros(num_classes)
            class_fn = np.zeros(num_classes)
            
            # 遍历所有样本
            for pred, gt in zip(predictions, ground_truths):
                pred_boxes = pred['boxes'].cpu().numpy()
                pred_labels = pred['labels'].cpu().numpy()
                pred_scores = pred['scores'].cpu().numpy()
                
                gt_boxes = gt['boxes'].cpu().numpy()
                gt_labels = gt['labels'].cpu().numpy()
                
                # 过滤低置信度预测
                mask = pred_scores >= threshold
                pred_boxes = pred_boxes[mask]
                pred_labels = pred_labels[mask]
                pred_scores = pred_scores[mask]
                
                # 如果都没有检测，跳过
                if len(pred_boxes) == 0 and len(gt_boxes) == 0:
                    continue
                
                # 计算 IoU 并匹配
                matched_gt = np.zeros(len(gt_boxes), dtype=bool)
                
                for pred_idx in range(len(pred_boxes)):
                    pred_label = pred_labels[pred_idx]
                    pred_box = pred_boxes[pred_idx]
                    
                    if len(gt_boxes) == 0:
                        # 预测有框但真实没有，FP
                        if pred_label > 0 and pred_label <= num_classes:
                            class_fp[pred_label - 1] += 1
                            all_fp += 1
                        continue
                    
                    # 计算 IoU
                    ious = self._compute_iou_matrix(
                        pred_box[np.newaxis, :], 
                        gt_boxes
                    )[0]
                    
                    # 找到最佳匹配
                    best_iou_idx = np.argmax(ious)
                    best_iou = ious[best_iou_idx]
                    
                    if best_iou >= 0.5 and not matched_gt[best_iou_idx]:
                        # 成功匹配
                        matched_gt[best_iou_idx] = True
                        gt_label = gt_labels[best_iou_idx]
                        
                        if pred_label == gt_label:
                            # TP
                            if pred_label > 0 and pred_label <= num_classes:
                                class_tp[pred_label - 1] += 1
                                all_tp += 1
                        else:
                            # 类别错误，算作 FP
                            if pred_label > 0 and pred_label <= num_classes:
                                class_fp[pred_label - 1] += 1
                                all_fp += 1
                    else:
                        # 没有匹配，FP
                        if pred_label > 0 and pred_label <= num_classes:
                            class_fp[pred_label - 1] += 1
                            all_fp += 1
                
                # 计算 FN（未匹配的 GT）
                for gt_idx in range(len(gt_boxes)):
                    if not matched_gt[gt_idx]:
                        gt_label = gt_labels[gt_idx]
                        if gt_label > 0 and gt_label <= num_classes:
                            class_fn[gt_label - 1] += 1
                            all_fn += 1
            
            # 计算 Precision 和 Recall
            for i in range(num_classes):
                p = class_tp[i] / (class_tp[i] + class_fp[i] + 1e-6)
                r = class_tp[i] / (class_tp[i] + class_fn[i] + 1e-6)
                pr_data[f'class_{i}']['precision'].append(float(p))
                pr_data[f'class_{i}']['recall'].append(float(r))
            
            # 综合指标
            p_all = all_tp / (all_tp + all_fp + 1e-6)
            r_all = all_tp / (all_tp + all_fn + 1e-6)
            pr_data['all_classes']['precision'].append(float(p_all))
            pr_data['all_classes']['recall'].append(float(r_all))
        
        # 计算每个类别的 AP
        for i in range(num_classes):
            prec = pr_data[f'class_{i}']['precision']
            rec = pr_data[f'class_{i}']['recall']
            pr_data[f'class_{i}']['ap'] = float(auc(rec, prec))
        
        # 计算综合 AP
        prec_all = pr_data['all_classes']['precision']
        rec_all = pr_data['all_classes']['recall']
        pr_data['all_classes']['ap'] = float(auc(rec_all, prec_all))
        
        # 添加类别名称映射
        pr_data['class_names'] = class_names
        
        return pr_data
    
    def plot_pr_curve(self, output_dir):
        """
        从 pr_curves_data.json 绘制分类别 PR 曲线（YOLOv8 风格）
        
        Args:
            output_dir: 输出目录路径
        
        Returns:
            matplotlib figure 或 None
        """
        json_path = Path(output_dir) / 'evaluation' / 'pr_curves_data.json'
        
        if not json_path.exists():
            print(f"Warning: PR curves data file not found: {json_path}")
            return None
        
        with open(json_path, 'r') as f:
            pr_data = json.load(f)
        
        class_names = pr_data.get('class_names', [])
        num_classes = len(class_names)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # 颜色映射
        colors = plt.cm.tab10(np.linspace(0, 1, num_classes))
        
        # 绘制每个类别的 PR 曲线
        for i in range(num_classes):
            prec = pr_data[f'class_{i}']['precision']
            rec = pr_data[f'class_{i}']['recall']
            ap = pr_data[f'class_{i}']['ap']
            ax.plot(rec, prec, color=colors[i], linewidth=1.5, alpha=0.7,
                   label=f'{class_names[i]} ({ap:.3f})')
        
        # 绘制综合 PR 曲线（加粗）
        prec_all = pr_data['all_classes']['precision']
        rec_all = pr_data['all_classes']['recall']
        ap_all = pr_data['all_classes']['ap']
        ax.plot(rec_all, prec_all, color='blue', linewidth=3, alpha=1.0,
               label=f'all classes ({ap_all:.3f} mAP@0.5)')
        
        ax.set_xlabel('Recall', fontsize=12)
        ax.set_ylabel('Precision', fontsize=12)
        ax.set_title('Precision-Recall Curve (Box)', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0), fontsize=10, framealpha=0.9)
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        
        plt.tight_layout()
        return fig


print("✓ DetectionEvaluator class defined successfully")